In [1]:
import json
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

C:\Users\Main\AppData\Roaming\Python\Python310\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
C:\Users\Main\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
templates

[{'id': 'bail_application',
  'name': 'Bail Application',
  'description': 'Application requesting bail for an accused person in a criminal case. Contains accused details, FIR information, police station, legal sections, case facts, grounds for bail, and prayer.',
  'document_type': 'criminal',
  'path': 'templates/bail_application.docx'},
 {'id': 'divorce_agreement',
  'name': 'Divorce Agreement',
  'description': 'Legal agreement related to divorce between husband and wife. Contains party details, marriage information, terms of separation, financial arrangements, and other agreed conditions.',
  'document_type': 'family',
  'path': 'templates/divorce_agreement.docx'},
 {'id': 'legal_notice',
  'name': 'Legal Notice',
  'description': 'Formal legal notice sent by one party to another regarding a legal dispute, demand, obligation, or alleged violation.',
  'document_type': 'general',
  'path': 'templates/legal_notice.docx'}]

In [27]:
with open("templates.json", "r", encoding="utf-8") as file:
    templates = json.load(file)

In [11]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [82]:
def select_template(user_request, chat_history):

    with open("templates.json", "r", encoding="utf-8") as file:
        templates = json.load(file)

    template_info = ""

    for template in templates:
        template_info += f"""
    ID: {template["id"]}
    Name: {template["name"]}
    Description: {template["description"]}
    Document Type: {template["document_type"]}
    Path: {template["path"]}

    """


    prompt = f"""
    You are a document template selection agent.

    The user wants to create a legal document.

    Available templates:

    {template_info}


    Chat history for last 2 chats of users:
    {chat_history}

    User request:
    {user_request}

    Choose the most appropriate template.

    Rules:
    1. Only select a template from the available templates.
    2. Do not invent a template.
    3. If none of the templates match, say NONE.
    4. Return the template ID and a short reason.

    Format:

    Template ID: <id or NONE>
    Reason: <reason>
    """


    response = llm.invoke(prompt)
    template_id = (
    response.content
    .split("Template ID:", 1)[1]
    .split("\n", 1)[0]
    .strip()
    .strip('"')
    .strip("'")
)


    for template in templates:
        if template["id"] == template_id:
            return template["path"]

    return None
    


In [83]:
user_request = """
I need divorce.
"""

chat_history = """
User: I need a legal document for divorce
"""

result = select_template(user_request, chat_history)

print("LLM RESULT:")
print(result)

LLM RESULT:
templates/divorce_agreement.docx


In [48]:
r = result.split("Template ID: ")[1].split("  \n")[0]
print(r)

divorce_agreement


In [49]:
for template in templates:
        if template["id"] == r:
            print(r)
            
        

divorce_agreement


In [16]:
def get_template_by_id(template_id):

    for template in templates:

        if template["id"] == template_id:
            return template

    return None

In [17]:
selected = get_template_by_id("bail_application")

print(selected["path"])

templates/bail_application.docx
